In [1]:
%load_ext autoreload
%autoreload 2

from solana.rpc.api import Client
from solana.rpc.async_api import AsyncClient
from solana.rpc.commitment import Commitment, Finalized
from solders.keypair import Keypair
from solders.pubkey import Pubkey
from solders.transaction import VersionedTransaction
from solders.rpc.responses import GetTransactionResp
from solders.message import to_bytes_versioned
from websockets import connect
import requests
import base64
import json
import defi
import asyncio
from websockets.exceptions import ConnectionClosed, WebSocketException

with open("keys.json") as f:
    keys = f.read()
keys = json.loads(keys)

helius_api = keys["Helius_API"]
helius_rpc = keys["Helius_RPC"]
private_key = keys["Private_Key"]
helius_ws = keys["Helius_WS"]

In [2]:
rpc_url = "https://api.mainnet-beta.solana.com"
client = Client(rpc_url)

In [11]:
mint = "8dQfKfmMhHTjXz9qmCRP4RunB6Y8ScerL6gru1Gmpump"
resp = client.get_account_info(Pubkey.from_string(mint), encoding="base64")

In [ ]:
resp

In [21]:
from solana.rpc.api import Client

def extract_mint_authority(byte_list):
    """Extract the mint authority from raw SPL Token mint account data."""
    if len(byte_list) < 36:
        raise ValueError("Invalid mint account data length")
    authority_bytes = byte_list[4:36]
    return Pubkey(bytes(authority_bytes))

# Setup
rpc_url = "https://api.mainnet-beta.solana.com"
client = Client(rpc_url)
mint = "8dQfKfmMhHTjXz9qmCRP4RunB6Y8ScerL6gru1Gmpump"

# Fetch account info
resp = client.get_account_info(Pubkey.from_string(mint))

# Extract mint authority
if resp.value:
    byte_list = resp.value.data  # Already a list of integers
    mint_authority = extract_mint_authority(byte_list)
    print("Mint authority:", mint_authority)
else:
    print("No account data found.")


Mint authority: TSLvdd1pWpHVjahSpsvCXUbgwsL3JAcvokwaKt1eokM


# Websockets - LP

In [ ]:
PROGRAMS_TO_MONITOR = [
    {
        "program_id": "675kPX9MHTjS2zt1qfr1NYHuzeLXfQM9H24wFSUt1Mp8",
        "instruction_filter": "initialize2: InitializeInstruction2",
        "label": "Raydium LP V4 Pool",
        "track": False
    },
    {
        "program_id": "pAMMBay6oceH9fJKBRHGP5D4bD4sWpmSwMn52FMfXEA",
        "instruction_filter": "initialize_pool",
        "label": "PumpFun AMM",
        "track": True
    },
    {
        "program_id": "CPMMoo8L3F4NbTegBCKVNunggL7H1ZpdTHKxQB5qKP1C",
        "instruction_filter": "initialize2: InitializeInstruction2",
        "label": "Raydium CPMM",
        "track": False
    }
]

In [ ]:
async def fetch_transaction_details(signature: str, client: AsyncClient, program_id: Pubkey, label: str):
    """Fetch and parse transaction details for a given signature."""
    try:
        tx = await client.get_transaction(signature, commitment=Finalized, max_supported_transaction_version=0)
        if not isinstance(tx, GetTransactionResp) or not tx.value:
            print(f"No transaction found for signature: {signature}")
            return

        # Extract transaction details
        tx_data = tx.value.transaction
        accounts = tx_data.message.account_keys
        logs = tx.value.meta.logs

        print(f"\nDetected Transaction for {label} (Program ID: {program_id})")
        print(f"Signature: https://explorer.solana.com/tx/{signature}")
        print(f"Accounts Involved: {[str(acc) for acc in accounts[:5]]}")  # Limit to first 5 for brevity
        print(f"Logs (first 3): {logs[:3]}")  # Show first few logs for inspection
        return logs, accounts
    except Exception as e:
        print(f"Error fetching transaction {signature}: {e}")
        return None, None

In [ ]:
async def subscribe_to_programs(websocket, programs):
    """Subscribe to Solana programs for log events."""
    for program in PROGRAMS_TO_MONITOR:
        if not program["track"]:
            continue
        print(f"Subscribing to program: {program['label']})")
        program_id = program["program_id"]
        subscription_request = {
            "jsonrpc": "2.0",
            "id": PROGRAMS_TO_MONITOR.index(program) + 1,
            "method": "logsSubscribe",
            "params": [
                {"mentions": [str(program_id)]},
                {"commitment": "finalized"}
            ]
        }
        try:
            await websocket.send(json.dumps(subscription_request))
            print(f"Subscribed to program: {program['label']} ({program_id})")
        except (ConnectionClosed, WebSocketException) as e:
            print(f"Failed to subscribe to {program['label']} ({program_id}): {e}")
            continue  # Skip to next program
        except Exception as e:
            print(f"Unexpected error subscribing to {program['label']} ({program_id}): {e}")
            continue

async def listen_for_events(websocket, client, programs):
    async for message in websocket:
        data = json.loads(message)
        if "result" in data and isinstance(data["result"], str):
            # Initial subscription confirmation
            print(f"Subscription ID: {data['result']}")
            continue
        if "params" in data and "result" in data["params"]:
            result = data["params"]["result"]
            signature = result["value"]["signature"]
            logs = result["value"]["logs"]

            # Find which program triggered this transaction
            for program in programs:
                program_id = program["program_id"]
                instruction_filter = program["instruction_filter"]
                label = program["label"]

                # Check if program ID is mentioned in accounts
                if str(program_id) in result["value"]["accounts"]:
                    # If a specific instruction filter is provided, check for it
                    if instruction_filter and any(instruction_filter in log for log in logs):
                        print(f"\nMatched {instruction_filter} for {label}")
                        logs, accounts = await fetch_transaction_details(signature, client, program_id, label)
                    elif not instruction_filter:
                        # If no filter, log all transactions for inspection
                        print(f"\nNo specific instruction filter for {label}, logging transaction")
                        logs, accounts = await fetch_transaction_details(signature, client, program_id, label)

In [12]:
async def monitor_programs(programs, helius_ws, rpc_url="https://api.mainnet-beta.solana.com"):
    """Monitor Solana programs with reconnection logic."""
    try:
        async with AsyncClient(rpc_url) as client:
            async with connect(helius_ws) as websocket:
                await subscribe_to_programs(websocket, programs)
                await listen_for_events(websocket, client, programs)
    except KeyboardInterrupt:
        print("Program stopped by user, closing connection...")
    except Exception as e:
        print(f"WebSocket connection failed: {e}")

In [16]:
x = ['a', 'b', 'c']
x.remove('b')
x

['a', 'c']

In [14]:
await asyncio.run(monitor_programs(PROGRAMS_TO_MONITOR, helius_ws))

RuntimeError: asyncio.run() cannot be called from a running event loop

# Jupiter Swaps

In [ ]:
client = Client(HELIUS_RPC_URL)
keypair = Keypair.from_base58_string(PRIVATE_KEY)
wallet_pubkey = keypair.pubkey()

In [ ]:
def get_jupiter_quote(input_mint: str, output_mint: str, amount: int, slippage_bps: int) -> dict:
    """Fetch a swap quote from Jupiter API."""
    params = {
        "inputMint": input_mint,
        "outputMint": output_mint,
        "amount": amount,
        "slippageBps": slippage_bps,
        "restrictIntermediateTokens": "true",
        "onlyDirectRoutes ": "true"
    }
    response = requests.get(JUPITER_QUOTE_API, params=params)
    response.raise_for_status()
    return response.json()

def get_jupiter_swap_transaction(quote_response: dict, wallet: str) -> dict:
    """Get serialized swap transaction from Jupiter API."""
    payload = {
        "quoteResponse": quote_response,
        "userPublicKey": wallet,
        "wrapAndUnwrapSol": True,  # Automatically handle WSOL wrapping/unwrapping
        "dynamicComputeUnitLimit": True,
        "prioritizationFeeLamports": {
              "priorityLevelWithMaxLamports": {
                  "maxLamports": 20000000, # 0.02 Sol
                  "global": True,
                  "priorityLevel": "veryHigh"
              }
          }
    }
    response = requests.post(JUPITER_SWAP_API, json=payload)
    response.raise_for_status()
    return response.json()

In [ ]:
try:
    buy_coin = "7oZcbyRE8nCLkwJ3pZnyrz9c4f6ucdC6RjUmrTHUCRaP"
    sell_coin = "So11111111111111111111111111111111111111112"
    amount = 
    buy_amount = sol_to_lamp(0.01)
    slippage = slip_to_bps(20)

    quote = get_jupiter_quote(sell_coin, buy_coin, AMOUNT, slippage)

    swap_tx = get_jupiter_swap_transaction(quote, str(wallet_pubkey))
    swap_instruction = swap_tx["swapTransaction"]

    # Step 3: Decode and sign the transaction
    raw_tx = VersionedTransaction.from_bytes(base64.b64decode(swap_instruction))    
    signature = keypair.sign_message(to_bytes_versioned(raw_tx.message))
    signed_tx = VersionedTransaction.populate(raw_tx.message, [signature])
    encoded_tx = base64.b64encode(bytes(signed_tx)).decode('utf-8')

    # Step 4: Send the transaction
    headers = {"Content-Type": "application/json"}
    data = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "sendTransaction",
        "params": [
            encoded_tx,
            {"skipPreflight": True, "encoding": "base64"}
        ]
    }
    response = requests.post(HELIUS_RPC_URL, headers=headers, json=data)
    response.raise_for_status()
    tx_id = response.json().get("result")
    print(f"Transaction sent: https://solscan.io/tx/{tx_id}")

except Exception as e:
    print(f"Error: {str(e)}")

Transaction sent: https://solscan.io/tx/4PfVpBdm2mNSQBZEQyXn5fmVtZb3Cw3RmfNHY53MGR7zH85N74ouEe4HAMoj96VmbxW9KSvMAWLDtfPTPEtYZEmp


In [44]:
response.json()

{'jsonrpc': '2.0',
 'result': '4PfVpBdm2mNSQBZEQyXn5fmVtZb3Cw3RmfNHY53MGR7zH85N74ouEe4HAMoj96VmbxW9KSvMAWLDtfPTPEtYZEmp',
 'id': 1}

In [37]:
response.json()

{'jsonrpc': '2.0',
 'error': {'code': -32602,
  'message': 'Invalid Request: ErrorObject { code: InvalidParams, message: "Invalid Request: invalid base58 encoding: InvalidCharacter { character: \'l\', index: 4 }", data: None }'},
 'id': 1}

In [ ]:
from solders.signature import Signature
tx_signature = Signature.from_string(tx_id)
tx_info = client.get_transaction(tx_signature, max_supported_transaction_version=0)
tx_info

In [58]:
tx_info.value.transaction.meta.post_token_balances[0]

UiTransactionTokenBalance(
    UiTransactionTokenBalance {
        account_index: 3,
        mint: "7oZcbyRE8nCLkwJ3pZnyrz9c4f6ucdC6RjUmrTHUCRaP",
        ui_token_amount: UiTokenAmount {
            ui_amount: Some(
                3515692.470566,
            ),
            decimals: 6,
            amount: "3515692470566",
            ui_amount_string: "3515692.470566",
        },
        owner: Some(
            "BJApez4geZSGd18vbmXw37CGanJxhdmLMZdkN5kTW2P7",
        ),
        program_id: Some(
            "TokenkegQfeZyiNwAJbNbGKPFXCWuBvf9Ss623VQ5DA",
        ),
    },
)

In [ ]:
post_balances = tx_info.value.transaction.meta.post_token_balances
    for balance in post_balances:
        if balance.mint == output_mint and balance.owner == wallet_pubkey:
            # Convert amount to human-readable (divide by 10^decimals)
            decimals = token.get_mint_info().decimals
            amount = balance.ui_token_amount.ui_amount
            if amount is None:
                raise ValueError("Amount not found in transaction balances")
            return amount

In [ ]:
memecoin = ""
buy_amount = sol_to_lamp(0.01)
slippage = slip_to_bps(20)

try:
    # Get swap quote
    quote = get_jupiter_quote(INPUT_MINT, memecoin, AMOUNT, slippage)

    # Get swap transaction
    swap_tx = get_jupiter_swap_transaction(quote, str(wallet_pubkey))
    swap_instruction = swap_tx["swapTransaction"]

    # Step 3: Decode and sign the transaction
    raw_tx = VersionedTransaction.from_bytes(base64.b64decode(swap_instruction))    
    signature = keypair.sign_message(to_bytes_versioned(raw_tx.message))
    signed_tx = VersionedTransaction.populate(raw_tx.message, [signature])
    encoded_tx = base64.b64encode(bytes(signed_tx)).decode('utf-8')

    # Step 4: Send the transaction
    headers = {"Content-Type": "application/json"}
    data = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "sendTransaction",
        "params": [
            encoded_tx,
            {"skipPreflight": True}
        ]
    }
    response = requests.post(HELIUS_RPC_URL, headers=headers, json=data)
    response.raise_for_status()
    tx_id = response.json().get("result")
    print(f"Transaction sent: https://solscan.io/tx/{tx_id}")

except Exception as e:
    print(f"Error: {str(e)}")